# 35 - description 召回回归测试

> 学习目标：用代码模拟 LLM 召回，写一个 RecallTester 跑 20+ query 验证 description 是否写好。
> 预备：33 / 34 跑过。

为什么重要：description 写完不动脑子 = 永远召不回。代码模拟能批量验证，写出回归测试套件。

In [ ]:
import os, shutil, re, json
from pathlib import Path
import yaml
from collections import Counter
import matplotlib.pyplot as plt

SBX = Path('./_skill_sandbox').resolve()
if SBX.exists():
    shutil.rmtree(SBX)
SBX.mkdir()
HOME_SKILLS = SBX / 'home' / '.claude' / 'skills'
HOME_SKILLS.mkdir(parents=True)
print('sandbox ready')

## 1. 「召回」的简单近似：description 里的词被用户 query 触发了多少

In [ ]:
def extract_keywords(text):
    quotes = re.findall(r'"([^"]+)"', text)
    neg_match = re.search(r'Do NOT trigger for?:\s*([^\n]+)', text, re.IGNORECASE)
    neg_phrases = []
    if neg_match:
        neg_text = neg_match.group(1)
        neg_phrases = re.findall(r'`([^`]+)`|"([^"]+)"', neg_text)
        neg_phrases = [a or b for a, b in neg_phrases]
    return {'quotes': set(quotes), 'neg': set(neg_phrases)}

def recall_score(query, skill_keywords):
    q_lower = query.lower()
    hits = set()
    for kw in skill_keywords['quotes']:
        if kw.lower() in q_lower:
            hits.add(kw)
    return len(hits), hits

desc = 'Trigger phrases include "explain this code", "what does this function do". Do NOT trigger for: rewriting code, reviewing PRs.'
kw = extract_keywords(desc)
print('keywords:', kw['quotes'])
for q in ['explain this code', 'open a PR', 'what does this function do', 'git status']:
    score, hits = recall_score(q, kw)
    print(f'  {q!r:30}  score={score}  hits={hits}')

## 2. 建一个小型 Skill 库 + RecallTester

In [ ]:
def make_skill(name, description):
    d = HOME_SKILLS / name
    d.mkdir(exist_ok=True)
    body = '# Steps\n1. do the thing\n2. verify\n\n# Example\nRun me\n'
    front = '---' + chr(10) + yaml.safe_dump({'name': name, 'description': description}, allow_unicode=True, sort_keys=False) + '---'
    (d / 'SKILL.md').write_text(front + chr(10) + chr(10) + body, encoding='utf-8')

make_skill('commit-pr',
           'When the user wants to commit and open a pull request, or says "commit", "create PR", "open a PR", "commit 一下". Do NOT trigger for: just running `git status` or `git log`.')
make_skill('explain-code',
           'When the user asks to explain a snippet of code, a function, or a class. Trigger: "explain this code", "what does this function do", "解读这段代码". Do NOT trigger for: rewriting code or reviewing PRs.')
make_skill('run-eval',
           'When the user wants to run an eval set or evaluation. Trigger: "run eval", "evaluate", "跑评估", "benchmark". Do NOT trigger for: ad-hoc single query testing.')
make_skill('send-slack',
           'When the user wants to send a Slack message or notification. Trigger: "send to slack", "slack 一下", "notify". Do NOT trigger for: reading Slack messages.')
print(f'created 4 skills')

In [ ]:
class RecallTester:
    def __init__(self, skills_root):
        self.skills_root = skills_root
        self.skills = {}
        for entry in sorted(skills_root.iterdir()):
            md = entry / 'SKILL.md'
            if md.is_file():
                text = md.read_text(encoding='utf-8')
                if text.startswith('---'):
                    end = text.find(chr(10) + '---', 3)
                    if end > 0:
                        fm = yaml.safe_load(text[3:end])
                        self.skills[fm.get('name', '?')] = {
                            'desc': fm.get('description', ''),
                            'kws': extract_keywords(fm.get('description', '')),
                        }

    def test(self, queries):
        results = []
        for q in queries:
            query, expect = q['query'], q['expect']
            scores = {}
            for name, info in self.skills.items():
                score, _ = recall_score(query, info['kws'])
                if score > 0:
                    scores[name] = score
            if scores:
                max_s = max(scores.values())
                recalled = [n for n, s in scores.items() if s == max_s]
            else:
                recalled = []
            correct = (any(r in expect for r in recalled) if expect else recalled == [])
            results.append({'query': query, 'expect': expect, 'recalled': recalled, 'scores': scores, 'correct': correct})
        return results

    def report(self, results):
        total = len(results)
        correct = sum(r['correct'] for r in results)
        recalls = Counter()
        for r in results:
            for n in r['recalled']:
                recalls[n] += 1
        false_pos = {}
        for r in results:
            if not r['expect'] and r['recalled']:
                for n in r['recalled']:
                    false_pos[n] = false_pos.get(n, 0) + 1
        return {
            'accuracy': correct / total if total else 0,
            'total': total, 'correct': correct,
            'recalls_per_skill': dict(recalls),
            'false_pos_per_skill': false_pos,
        }

In [ ]:
test_queries = [
    {'query': '帮我 commit 一下',         'expect': ['commit-pr']},
    {'query': 'create PR',                'expect': ['commit-pr']},
    {'query': '提 PR',                    'expect': ['commit-pr']},
    {'query': 'open a pull request',     'expect': ['commit-pr']},
    {'query': 'commit these changes',    'expect': ['commit-pr']},
    {'query': 'explain this code',         'expect': ['explain-code']},
    {'query': '解读这段代码',            'expect': ['explain-code']},
    {'query': 'what does this function do', 'expect': ['explain-code']},
    {'query': 'explain the class',       'expect': ['explain-code']},
    {'query': 'walk me through this',     'expect': []},
    {'query': 'run eval',                 'expect': ['run-eval']},
    {'query': '跑评估',                  'expect': ['run-eval']},
    {'query': 'benchmark the system',    'expect': []},
    {'query': 'send to slack',            'expect': ['send-slack']},
    {'query': 'notify on slack',          'expect': ['send-slack']},
    {'query': 'slack 一下',              'expect': ['send-slack']},
    {'query': 'git status',               'expect': []},
    {'query': 'rewriting this function', 'expect': []},
    {'query': 'add a new test',          'expect': []},
    {'query': 'what is the weather',     'expect': []},
    {'query': 'tell me a joke',           'expect': []},
]
tester = RecallTester(HOME_SKILLS)
results = tester.test(test_queries)
report = tester.report(results)
print(f'test set: {len(test_queries)} queries')
print(f'\naccuracy: {report["accuracy"]*100:.1f}%  ({report["correct"]}/{report["total"]})')
print(f'recalls per skill: {report["recalls_per_skill"]}')
print(f'false positive:    {report["false_pos_per_skill"]}')

In [ ]:
print('=' * 60)
print('specific case (pass/fail):')
print('=' * 60)
for r in results:
    icon = 'pass' if r['correct'] else 'fail'
    recalled = r['recalled'] or 'NONE'
    expect = r['expect'] or 'NONE'
    print(f'{icon}  {r["query"]!r:40}  expect={expect!s}  recalled={recalled}')

## 3. 改 description → 重测 → 看召回率提升

In [ ]:
make_skill('run-eval',
           'When the user wants to run an eval set or evaluation. Trigger: "run eval", "evaluate", "跑评估", "benchmark", "benchmark the system", "evaluation suite". Do NOT trigger for: ad-hoc single query testing.')
make_skill('explain-code',
           'When the user asks to explain a snippet of code, a function, or a class. Trigger: "explain this code", "what does this function do", "解读这段代码", "walk me through". Do NOT trigger for: rewriting code or reviewing PRs.')
make_skill('send-slack',
           'When the user wants to send a Slack message or notification. Trigger: "send to slack", "slack 一下", "notify", "slack the team". Do NOT trigger for: reading Slack messages.')

tester2 = RecallTester(HOME_SKILLS)
results2 = tester2.test(test_queries)
report2 = tester2.report(results2)
print(f'after description fix accuracy: {report2["accuracy"]*100:.1f}%  ({report2["correct"]}/{report2["total"]})')
print(f'false positive: {report2["false_pos_per_skill"]}')

In [ ]:
shutil.rmtree(SBX, ignore_errors=True)
print('sandbox cleaned')

## 深入思考

1. 为什么不直接用 LLM 测？便宜 + 可复现，但抓不住语义。
2. 描述加 Do NOT 反例 keyword 怎么写？暂无 — 留作 TODO。
3. 多语言测试集：description 双语（中英）= 召回集也双语言。
4. description 太长（>500 字符）= 误召变多。
5. 3 个月不被召的 Skill = 该砍。

改一改：把 false_pos 检验加上（Do NOT 反例 keyword 命中 → 必扣分）。

## 自检

- [ ] 解释为什么关键词匹配是 LLM 召回的「下限测试」
- [ ] 默写 RecallTester 召回逻辑
- [ ] 解释 false_pos 比 false_neg 更要重视
- [ ] 给一个 Skill 召回率低，能立刻判断是 description 含糊还是 trigger phrases 缺

下一步: 36_skill_subagent_compose.ipynb